# ReAct Harness Architecture and Demo

This notebook documents the harness architecture in `agent_react/react_agent.py`, explains the current interaction flow, and shows a concrete prompt example for the current design.

## Architecture Overview

The harness is designed around several modular components:

1. `BaseAgentHarness` — the generic runtime loop for constructing prompts, calling the model, parsing responses, validating actions, executing tools, and continuing reasoning.
2. `AgentPattern` / `ReActPattern` — the interaction pattern abstraction that defines how models should format requests and how responses are parsed.
3. `ToolRegistry` / `SkillRegistry` — metadata registries for executable tools and available skills.
4. Dynamic loading — skills are loaded from markdown files in `skills/`, and tools are loaded from YAML + Python files in `tools/`.

This structure keeps the core harness reusable, while the current demo uses a ReAct-style pattern implementation. It also makes it easier to add future patterns without changing the base loop.

## Current prompt construction

`build_prompt()` now merges: 
- the static system prompt from `system_prompt.md`
- static memory guidance
- tool descriptions generated from `self.tools`
- skill descriptions generated from `self.skills`
- recent conversation history from `MemoryStore`
- the current user input

In [ ]:
from react_agent import create_default_agent

agent = create_default_agent()
prompt_messages = agent.build_prompt(
    'What is ReAct?'
)
for message in prompt_messages:
    print(message['role'])
    print(message['content'])
    print('---')

## Interaction flow

A simplified example of the runtime flow is: 

1. User sends a query.
2. The harness builds a prompt that includes system instructions, tool descriptions, skill descriptions, and memory.
3. The model returns a response in the format:
   - `Selected skill: <skill_name>`
   - `Action: <tool_name>`
   - `Action Input: <input>`
4. The harness parses the model output and validates the skill/tool.
5. The harness executes the tool implementation.
6. Tool output is stored in `MemoryStore`.
7. The harness prompts the model again with the tool result if more reasoning is required.

## Improvements made

- Prompt is now generated dynamically from `self.tools` and `self.skills` instead of cached context tool/skill entries.
- `MemoryStore` history is included in the prompt so tool outputs and previous reasoning are visible to the model.
- The run loop now passes a follow-up prompt after tool execution, instead of reusing the previous raw model response as input.
- Added validation for skill/tool selection and retry behavior when the model output is malformed.
- Tools are now loaded from `tools/*.yaml` + `tools/*.py`, supporting extensible tool plugins.

## Potential future improvements

- Add a dedicated message channel for observations instead of packing memory into a system message.
- Support tool input schemas and examples to make tool selection more reliable.
- Persist longer conversation history separately from the limited `MemoryStore` buffer.
- Add a more robust parse-and-retry loop with explicit formatting correction.
- Provide a tool specification interface so the model can reason with structured tool signatures.